In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/typing.py:47: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/typing.py:101: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
cora_dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = cora_dataset[0]
data = data.to(device)

Processing...
Done!


In [3]:
in_feats = data.x.shape[1]
h_channels = 64
heads = 8
model = GAT(cora_dataset.num_features, h_channels, cora_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
train(model, data, data.train_mask, data.y)

1.9456170797348022

In [5]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.9325, Accuracy: 0.5230
Epoch: 001, Loss: 1.9131, Accuracy: 0.6100
Epoch: 002, Loss: 1.8900, Accuracy: 0.6930
Epoch: 003, Loss: 1.8744, Accuracy: 0.7510
Epoch: 004, Loss: 1.8487, Accuracy: 0.7730
Epoch: 005, Loss: 1.8264, Accuracy: 0.7640
Epoch: 006, Loss: 1.8188, Accuracy: 0.7800
Epoch: 007, Loss: 1.7611, Accuracy: 0.7910
Epoch: 008, Loss: 1.7520, Accuracy: 0.8100
Epoch: 009, Loss: 1.7510, Accuracy: 0.8090
Epoch: 010, Loss: 1.7013, Accuracy: 0.8020
Epoch: 011, Loss: 1.6481, Accuracy: 0.8070
Epoch: 012, Loss: 1.6351, Accuracy: 0.8200
Epoch: 013, Loss: 1.5905, Accuracy: 0.8230
Epoch: 014, Loss: 1.6015, Accuracy: 0.8070
Epoch: 015, Loss: 1.5256, Accuracy: 0.7930
Epoch: 016, Loss: 1.5119, Accuracy: 0.7890
Epoch: 017, Loss: 1.4602, Accuracy: 0.7930
Epoch: 018, Loss: 1.4730, Accuracy: 0.7980
Epoch: 019, Loss: 1.3728, Accuracy: 0.8050
Epoch: 020, Loss: 1.3589, Accuracy: 0.8030
Epoch: 021, Loss: 1.2885, Accuracy: 0.8070
Epoch: 022, Loss: 1.3105, Accuracy: 0.8130
Epoch: 023,

In [6]:
torch.save(model.state_dict(), 'cora_gat.pt')